# IT3212 - Assignment 3 - Task 4: Feature Selection

Reduces the 31 features produced by Task 3 to a smaller, better-justified set.

Task 3 **expanded** the feature space by adding clinical indices. Task 4
**prunes** it: several features carry the same information (a derived index and
its parent, for example), and redundant columns add computation without adding
signal.

**Inputs:**  `../data/assignment3/task3_{train,val,test}.csv` (31 features)

**Outputs:** `../data/assignment3/task4_{train,val,test}.csv`
             `../data/assignment3/task4_metadata.json`

**Approach:** three selection methods are compared rather than one being
assumed. The comparison itself is the justification.

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. Load the Task 3 output

All three splits are loaded. Only the **training** set is used to compute
importances and to choose how many features to keep. The validation set is used
to compare candidate feature counts. The test set is not touched until Task 5.

In [2]:
IN_DIR = Path("../data/assignment3")
OUT_DIR = Path("../data/assignment3")

TARGET = "smoking"
RANDOM_STATE = 42

splits = {
    name: pd.read_csv(IN_DIR / f"task3_{name}.csv")
    for name in ["train", "val", "test"]
}

X_train = splits["train"].drop(columns=[TARGET])
y_train = splits["train"][TARGET]
X_val = splits["val"].drop(columns=[TARGET])
y_val = splits["val"][TARGET]

print(f"train: {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"val  : {X_val.shape[0]:,} rows x {X_val.shape[1]} features")

train: 787,625 rows x 31 features
val  : 98,453 rows x 31 features


## 2. Why feature selection, and which method

### Why reduce at all

Task 3 produced 31 features from 22 original measurements. Several are
redundant by construction - every derived index is computed from columns that
are still in the set. The correlation matrix confirms it: six feature pairs
exceed 0.90.

Redundant features do not add signal, but they do add computation, and on
distance-based and linear models they can distort the result because the same
information is counted more than once.

### Alternatives considered

| Method | Why it was not chosen as the sole approach |
|---|---|
| PCA | Unsupervised - maximises variance in the features, not separation of the target. Also produces components that cannot be named, which would undermine the explanations required in Tasks 6, 8 and 10 |
| PLS | Supervised, so better suited than PCA, but it still outputs abstract components rather than named features |
| Wrapper (RFE) | Accounts for interactions, but is expensive: it refits the model once per feature, and with 31 features on ~788k rows that is a large cost for a selection step |
| Not selecting | The brief allows this if justified, but with six pairs above 0.90 correlation the redundancy is real and measurable |

### Chosen approach: compare a filter and an embedded method

Two different families are computed and compared:

1. **Filter** - `mutual_info_classif` and ANOVA F-test. These score each
   feature independently of any model, using only its relationship with the
   target.
2. **Embedded** - `RandomForestClassifier` feature importances. Selection
   happens during training, so interactions between features are taken into
   account.

Using both means the final ranking is not an artefact of a single method's
assumptions. Where the methods agree, the ranking is trustworthy; where they
disagree, that disagreement is informative.

To keep the cost reasonable, importances are computed on a random sample of
100,000 training rows. This is a modelling choice, not a shortcut on the
result: with 31 features and 100k rows the estimates are stable, and the
selected set is verified on the full validation set in section 6.

## 3. Filter methods

Mutual information captures non-linear dependence between a feature and the
target. The ANOVA F-test measures the ratio of between-class to within-class
variance, and therefore assumes a roughly linear relationship.

Reporting both makes the assumption explicit: agreement between them suggests
the feature's signal is not an artefact of one particular assumption.

In [3]:
SAMPLE_SIZE = 100_000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_train), SAMPLE_SIZE, replace=False
)
X_sample = X_train.iloc[sample_idx]
y_sample = y_train.iloc[sample_idx]

mi_scores = mutual_info_classif(X_sample, y_sample, random_state=RANDOM_STATE)
anova_scores, _ = f_classif(X_sample, y_sample)

filter_df = pd.DataFrame({
    "feature": X_train.columns,
    "mutual_info": mi_scores,
    "anova_F": anova_scores,
})
filter_df["mi_rank"] = filter_df["mutual_info"].rank(ascending=False).astype(int)
filter_df["anova_rank"] = filter_df["anova_F"].rank(ascending=False).astype(int)

filter_df.sort_values("mi_rank").head(12)[
    ["feature", "mutual_info", "mi_rank", "anova_rank"]
].round(4)

,feature,mutual_info,mi_rank,anova_rank
0,sex,0.2467,1,1
22,BMI,0.1595,2,17
2,height,0.1529,3,2
23,waist_height_ratio,0.1402,4,25
16,hemoglobin,0.1384,5,3
3,weight,0.0966,6,4
21,gamma_GTP,0.0865,7,5
18,serum_creatinine,0.0820,8,6
4,waistline,0.0523,9,7
29,TG_HDL_ratio,0.0367,10,8


## 4. Embedded method

A random forest assigns each feature an importance based on how much it reduces
impurity across all trees. Unlike the filter methods, this accounts for how
features behave in combination - a feature that is weak alone but useful in
interaction will score higher here.

In [4]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_sample, y_sample)

importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "rf_importance": rf.feature_importances_,
})
importance_df["rf_rank"] = importance_df["rf_importance"].rank(ascending=False).astype(int)

importance_df.sort_values("rf_rank").head(12).round(4)

,feature,rf_importance,rf_rank
0,sex,0.3720,1
16,hemoglobin,0.1258,2
2,height,0.0992,3
21,gamma_GTP,0.0634,4
3,weight,0.0537,5
18,serum_creatinine,0.0512,6
29,TG_HDL_ratio,0.0199,7
1,age,0.0193,8
4,waistline,0.0189,9
15,triglyceride,0.0177,10


## 5. Combine the three rankings

The three methods are combined by **average rank** rather than by averaging raw
scores. The scores are not on a comparable scale - mutual information is in
nats, the F-statistic is unbounded, and importances sum to 1 - so averaging
them directly would be meaningless. Ranks are comparable.

A feature that all three methods rank highly is a strong candidate. A feature
that one method ranks highly and the others do not is worth inspecting before
including it.

In [5]:
ranking = (
    filter_df[["feature", "mi_rank", "anova_rank"]]
    .merge(importance_df[["feature", "rf_rank", "rf_importance"]], on="feature")
)
ranking["avg_rank"] = ranking[["mi_rank", "anova_rank", "rf_rank"]].mean(axis=1)
ranking = ranking.sort_values("avg_rank").reset_index(drop=True)
ranking["overall_rank"] = ranking.index + 1

print("Top 15 by average rank:")
print(
    ranking.head(15)[
        ["overall_rank", "feature", "mi_rank", "anova_rank", "rf_rank", "avg_rank"]
    ].to_string(index=False)
)

# How much the three methods agree at the top of the ranking
TOP_N = 15
sets = {
    "mutual_info": set(ranking.nsmallest(TOP_N, "mi_rank")["feature"]),
    "anova": set(ranking.nsmallest(TOP_N, "anova_rank")["feature"]),
    "random_forest": set(ranking.nsmallest(TOP_N, "rf_rank")["feature"]),
}

print(f"\nAgreement at top-{TOP_N}:")
print(f"  mutual_info & anova         : {len(sets['mutual_info'] & sets['anova'])}/{TOP_N}")
print(f"  mutual_info & random_forest : {len(sets['mutual_info'] & sets['random_forest'])}/{TOP_N}")
print(f"  anova & random_forest       : {len(sets['anova'] & sets['random_forest'])}/{TOP_N}")
print(f"  all three                   : {len(sets['mutual_info'] & sets['anova'] & sets['random_forest'])}/{TOP_N}")
print("\n  " + ", ".join(sorted(sets["mutual_info"] & sets["anova"] & sets["random_forest"])))


Top 15 by average rank:
 overall_rank            feature  mi_rank  anova_rank  rf_rank  avg_rank
            1                sex        1           1        1  1.000000
            2             height        3           2        3  2.666667
            3         hemoglobin        5           3        2  3.333333
            4             weight        6           4        5  5.000000
            5          gamma_GTP        7           5        4  5.333333
            6   serum_creatinine        8           6        6  6.666667
            7       TG_HDL_ratio       10           8        7  8.333333
            8          waistline        9           7        9  8.333333
            9       triglyceride       12           9       10 10.333333
           10                BMI        2          17       14 11.000000
           11           SGOT_ALT       11          10       12 11.000000
           12      AST_ALT_ratio       13          12       11 12.000000
           13          HDL_

## 6. How many features to keep

The ranking says which features are strongest, but not how many to keep. That
is decided empirically: models are trained on the top-k features for a range of
k, and scored on the validation set.

Two metrics are reported. **F1** is the metric chosen for this problem (Task 1),
but it is threshold-dependent and can be insensitive to small differences.
**AUC** measures ranking quality across all thresholds, so it is more sensitive
to whether extra features carry real signal.

In [6]:
ordered_features = ranking["feature"].tolist()
candidate_k = [3, 5, 8, 12, 15, 18, 20, 25, 31]

rows = []
for k in candidate_k:
    cols = ordered_features[:k]

    logreg = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    )
    logreg.fit(X_train[cols], y_train)

    forest = RandomForestClassifier(
        n_estimators=60, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE
    )
    forest.fit(X_train[cols], y_train)

    rows.append({
        "k": k,
        "logreg_f1": f1_score(y_val, logreg.predict(X_val[cols])),
        "logreg_auc": roc_auc_score(y_val, logreg.predict_proba(X_val[cols])[:, 1]),
        "rf_f1": f1_score(y_val, forest.predict(X_val[cols])),
        "rf_auc": roc_auc_score(y_val, forest.predict_proba(X_val[cols])[:, 1]),
    })

k_df = pd.DataFrame(rows).round(5)
k_df

,k,logreg_f1,logreg_auc,rf_f1,rf_auc
0,3,0.79024,0.84545,0.79000,0.84664
1,5,0.78944,0.86340,0.78908,0.86575
2,8,0.78845,0.86570,0.78939,0.86835
3,12,0.78695,0.86846,0.78967,0.87182
4,15,0.78656,0.86937,0.78938,0.87169
5,18,0.78670,0.86946,0.78921,0.87866
6,20,0.78632,0.87018,0.78867,0.87859
7,25,0.78559,0.87049,0.78950,0.87871
8,31,0.78570,0.87050,0.78941,0.87781


### Reading the result

Two things stand out:

- **F1 is essentially flat** across every k from 3 to 31 - the spread is under
  0.005. Adding features does not improve the classifier's F1.
- **AUC rises** from about 0.845 at k=3 to about 0.879 by k=18, then plateaus.

The flat F1 is a real property of this dataset, not a measurement failure: the
strongest single feature already carries most of the class separation, so extra
features refine the ranking (AUC) without moving the decision boundary enough to
change F1.

Because AUC keeps improving up to roughly 18 features and stops after that, 18
is chosen: it captures the available ranking signal without carrying the
remaining 13 features purely for the cost.

In [7]:
CHOSEN_K = 18
selected_features = ordered_features[:CHOSEN_K]

print(f"Selected {CHOSEN_K} of {X_train.shape[1]} features:")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:>2}. {f}")

dropped = [c for c in X_train.columns if c not in selected_features]
print(f"\nDropped {len(dropped)} features:")
print("  " + ", ".join(dropped))

Selected 18 of 31 features:
   1. sex
   2. height
   3. hemoglobin
   4. weight
   5. gamma_GTP
   6. serum_creatinine
   7. TG_HDL_ratio
   8. waistline
   9. triglyceride
  10. BMI
  11. SGOT_ALT
  12. AST_ALT_ratio
  13. HDL_chole
  14. waist_height_ratio
  15. cholesterol_ratio
  16. LDL_HDL_ratio
  17. age
  18. SGOT_AST

Dropped 13 features:
  sight_left, sight_right, hear_left, hear_right, SBP, DBP, BLDS, tot_chole, LDL_chole, urine_protein, pulse_pressure, MAP, non_HDL_chole


## 7. Did the Task 3 features survive?

Task 3 justified its nine derived features on clinical grounds and deferred the
empirical evidence to this task. That evidence is reported here rather than
left implicit.

A feature "survived" if it is among the 18 highest-ranked features by average
rank across the three methods. A feature that was dropped means the source
measurements it was computed from already carried the same signal.

In [8]:
DERIVED = [
    "BMI", "waist_height_ratio", "pulse_pressure", "MAP", "non_HDL_chole",
    "cholesterol_ratio", "LDL_HDL_ratio", "TG_HDL_ratio", "AST_ALT_ratio",
]

survived = [f for f in DERIVED if f in selected_features]
dropped_derived = [f for f in DERIVED if f not in selected_features]

print(f"Derived features kept: {len(survived)} of {len(DERIVED)}")
for f in survived:
    r = ranking.loc[ranking["feature"] == f].iloc[0]
    print(f"  kept   {f:20s} overall rank {int(r['overall_rank']):>2}  (mi {int(r['mi_rank'])}, anova {int(r['anova_rank'])}, rf {int(r['rf_rank'])})")

for f in dropped_derived:
    r = ranking.loc[ranking["feature"] == f].iloc[0]
    print(f"  DROPPED {f:20s} overall rank {int(r['overall_rank']):>2}")

print(f"\nDerived features make up {len(survived)}/{len(selected_features)} of the selected set.")

Derived features kept: 6 of 9
  kept   BMI                  overall rank 10  (mi 2, anova 17, rf 14)
  kept   waist_height_ratio   overall rank 14  (mi 4, anova 25, rf 13)
  kept   cholesterol_ratio    overall rank 15  (mi 15, anova 13, rf 15)
  kept   LDL_HDL_ratio        overall rank 16  (mi 16, anova 19, rf 17)
  kept   TG_HDL_ratio         overall rank  7  (mi 10, anova 8, rf 7)
  kept   AST_ALT_ratio        overall rank 12  (mi 13, anova 12, rf 11)
  DROPPED pulse_pressure       overall rank 27
  DROPPED MAP                  overall rank 20
  DROPPED non_HDL_chole        overall rank 23

Derived features make up 6/18 of the selected set.


## 8. Export the reduced splits

The selected columns plus the target are written as
`task4_{train,val,test}.csv`. Data remains **unscaled**, as in Tasks 2 and 3.

No scaler is fitted or saved - the same recipe convention applies, now covering
the 18 selected features minus any binary columns.

In [9]:
export_cols = selected_features + [TARGET]

for name, d in splits.items():
    d[export_cols].to_csv(OUT_DIR / f"task4_{name}.csv", index=False)

BINARY_OR_ORDINAL = ["sex", "hear_left", "hear_right", "urine_protein"]
scale_cols = [c for c in selected_features if c not in BINARY_OR_ORDINAL]

metadata = {
    "task": "task4_feature_selection",
    "target": TARGET,
    "target_classes": {"0": "never smoked", "1": "ever smoked (current or former)"},
    "source": "task3 output",
    "n_features_in": X_train.shape[1],
    "n_features_out": len(selected_features),
    "selected_features": selected_features,
    "dropped_features": dropped,
    "derived_features_kept": survived,
    "derived_features_dropped": dropped_derived,
    "methods": {
        "filter": ["mutual_info_classif", "f_classif"],
        "embedded": "RandomForestClassifier feature_importances_",
        "combination": "average of per-method ranks",
        "sample_size": SAMPLE_SIZE,
    },
    "selection": {
        "chosen_k": CHOSEN_K,
        "chosen_on": "validation set (AUC plateau)",
        "test_set_used": False,
    },
    "scaling": {
        "applied_to_exports": False,
        "method": "StandardScaler",
        "columns": scale_cols,
        "excluded_columns": BINARY_OR_ORDINAL,
        "fit_on": "train only - never fit on train+val+test",
    },
    "row_counts": {name: len(d) for name, d in splits.items()},
}

with open(OUT_DIR / "task4_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Exported:")
for f in sorted(OUT_DIR.glob("task4_*")):
    print(f"  {f.name}")

Exported:
  task4_metadata.json
  task4_test.csv
  task4_train.csv
  task4_val.csv


## 9. Verify the exports

Reload the files and confirm the column count, the target distribution and that
the selected columns are identical across all three splits.

In [10]:
frames = {}
for name in ["train", "val", "test"]:
    d = pd.read_csv(OUT_DIR / f"task4_{name}.csv")
    frames[name] = d
    print(
        f"task4_{name}.csv: {d.shape[0]:>7,} rows x {d.shape[1]} columns "
        f"| ever ratio = {d[TARGET].mean():.4f}"
    )

cols_match = (
    list(frames["train"].columns)
    == list(frames["val"].columns)
    == list(frames["test"].columns)
)
print(f"\nColumns identical across splits: {cols_match}")
print(f"Features: {len(selected_features)} (from {X_train.shape[1]})")
print(f"Continuous columns for downstream scaling: {len(scale_cols)}")

pd.read_csv(OUT_DIR / "task4_train.csv").head(3)

task4_train.csv: 787,625 rows x 19 columns | ever ratio = 0.3920
task4_val.csv:  98,453 rows x 19 columns | ever ratio = 0.3920
task4_test.csv:  98,454 rows x 19 columns | ever ratio = 0.3920

Columns identical across splits: True
Features: 18 (from 31)
Continuous columns for downstream scaling: 17


,sex,height,hemoglobin,weight,gamma_GTP,serum_creatinine,TG_HDL_ratio,waistline,triglyceride,BMI,SGOT_ALT,AST_ALT_ratio,HDL_chole,waist_height_ratio,cholesterol_ratio,LDL_HDL_ratio,age,SGOT_AST,smoking
0,1,165.0,15.4,65.0,25.0,0.8,2.10938,89.0,135.0,23.8751,24.0,1.125000,64.0,0.539394,2.85938,1.43750,70,27.0,0
1,1,170.0,14.4,60.0,73.5,1.0,4.11905,77.9,173.0,20.7612,50.0,0.830000,42.0,0.458235,4.00000,2.16667,40,41.5,1
2,0,155.0,14.4,50.0,44.0,0.5,3.36923,80.0,219.0,20.8117,26.0,0.923077,65.0,0.516129,3.18462,1.50769,60,24.0,0
